# 16. In Progress: Draft-Day Matchup Tools

Part of the `nuclearff` [User Tutorial](https://nolmacdonald.github.io/nuclearff/tutorial/index.html) notebook series -- every notebook shares one running `./demo` directory and depends on the ones before it having already run. See [README.md](README.md) for the full run order. Run `15_wins_and_leagues.ipynb` first.

`nuclearff.matchups` and `nuclearff.draft` are the start of a larger, still-unfinished feature area -- a live draft companion. Only two functions exist so far, each real and independently useful, but neither is wired into a report, a visualization, or the CLI yet. This notebook covers what's actually there today. Mirrors the [Draft Companion Tools](https://nolmacdonald.github.io/nuclearff/tutorial/16_draft_companion_tools.html) docs page.

**Both functions below are building blocks, not a finished feature** -- there is no report, no CLI command, and no visualization built on top of either yet, and the wider draft-companion epic they belong to ([GitHub issue #102](https://github.com/nolmacdonald/nuclearff/issues/102)) covers a live draft-pick recommendation tool this is only the first piece of. Nothing here should be read as a finished, supported feature the way Chapters 5 through 15 are -- check the project's GitHub issues for current status before building on it.

## Defense-vs-position

`points_allowed_by_position` aggregates real fantasy points allowed, by position, per NFL defense -- the raw signal behind "this team is a soft matchup for wide receivers":

In [1]:
from nuclearff.config import load_league_config
from nuclearff.matchups.dvp import points_allowed_by_position
from nuclearff.nflverse import load_weekly_skill_stats
from nuclearff.scoring import ScoringEngine

league_cfg = load_league_config("./demo/configs/leagues/1367225133634191360.yaml")
engine = ScoringEngine(league_cfg.scoring)

weekly = load_weekly_skill_stats([2024])
dvp = points_allowed_by_position(weekly, engine)
dvp.filter(dvp["position"] == "WR").sort("points_allowed", descending=True).head(3)

ScoringEngine has no mapping for 48 nonzero scoring key(s); these will not be counted toward fantasy points: blk_kick=2.0, blk_kick_ret_yd=0.04, def_3_and_out=0.25, def_4_and_stop=0.5, def_pr_yd=0.04, def_st_ff=1.0, def_st_fum_rec=1.0, def_st_td=6.0, def_td=6.0, ff=1.0, fg_ret_yd=0.04, fgm_0_19=3.0, fgm_20_29=3.0, fgm_30_39=3.0, fgm_40_49=4.0, fgm_50p=5.0, fgmiss=-1.0, fgmiss_0_19=-1.0, fgmiss_20_29=-1.0, fum_rec=2.0, fum_rec_td=6.0, fum_ret_yd=0.04, int=2.0, int_ret_yd=0.04, kr_yd=0.04, pr_yd=0.04, pts_allow_0=5.0, pts_allow_14_20=1.0, pts_allow_1_6=4.0, pts_allow_28_34=-1.0, pts_allow_35p=-4.0, pts_allow_7_13=3.0, sack=1.0, safe=2.0, st_ff=1.0, st_fum_rec=1.0, st_td=6.0, tkl_loss=0.25, xpm=1.0, xpmiss=-1.0, yds_allow_0_100=5.0, yds_allow_100_199=3.0, yds_allow_200_299=2.0, yds_allow_350_399=-1.0, yds_allow_400_449=-3.0, yds_allow_450_499=-5.0, yds_allow_500_549=-6.0, yds_allow_550p=-7.0


season,week,opponent_team,position,points_allowed,season_avg_points_allowed
i32,i32,str,str,f64,f64
2024,5,"""TB""","""WR""",92.2,36.466667
2024,5,"""BAL""","""WR""",75.5,35.783158
2024,11,"""JAX""","""WR""",73.6,36.644706


Points allowed are scored under this league's own rules (`05_scoring_engine.ipynb`), not a generic formula -- the same real game would score differently under a half-PPR league.

## Strength of schedule

`strength_of_schedule` averages a team's *remaining* opponents' points-allowed at a position -- how favorable a draft candidate's rest-of-season schedule looks, given where they play:

In [2]:
from nuclearff.draft.sos import strength_of_schedule
from nuclearff.nflverse.schedules import load_schedules

schedules = load_schedules([2024])
sos = strength_of_schedule(
    "SF", "WR", dvp, schedules, 2024, remaining_weeks=[15, 16, 17]
)
print(f"49ers WR strength of schedule, weeks 15-17: {sos:.1f}")

49ers WR strength of schedule, weeks 15-17: 33.0


A higher number means those real remaining opponents allowed more real fantasy points to opposing WRs on average that season -- a softer remaining schedule for a wide receiver on that team than a lower number would indicate.

> **Why this matters:** it's worth including a not-yet-finished feature area in a tutorial at all, rather than waiting until it's polished -- the two functions above are exactly as real and load-bearing as anything in Chapter 6's metrics, just not wired into a report yet. If you're extending this project yourself, this is a good place to see what "library primitive with no UI yet" looks like in practice, before deciding whether to build the report/CLI layer on top yourself.

## What's Next

`17_querying_and_provenance.ipynb` closes out this tutorial: one shared database for everything built across every chapter, and how to record exactly how a given dataset or report was produced.

## See Also

- `05_scoring_engine.ipynb` -- the `ScoringEngine` this chapter's DVP metric depends on.
- [Draft Companion Tools](https://nolmacdonald.github.io/nuclearff/tutorial/16_draft_companion_tools.html) -- the matching docs page.
- [GitHub issue #102](https://github.com/nolmacdonald/nuclearff/issues/102) -- the wider draft-companion epic.